# Sosyal Muhendislik Metinlerinde Birinci Dereceden Mantik Cikarimi
Deney 4 - Yapay Zeka Dersi

## 1. Veri Setinin Yuklenmesi ve FOL Olgularina Donusturulmesi

In [1]:
import csv, time, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIELDS = ["MessageID","Channel","SenderKnown","DomainMatch","HasLink","ShortenedURL",
          "RequestsCredential","RequestsPayment","Urgency","AuthorityClaim","Attachment",
          "MacroRisk","PriorThread","Label"]

with open("messages.csv", newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

len(rows)


14

In [1]:
def row_to_facts(row):
    m = row["MessageID"]
    facts = set()
    facts.add(("Message",m))
    ch = row["Channel"]
    if ch == "Email": facts.add(("Email",m))
    if ch == "SMS": facts.add(("SMS",m))
    if ch == "VoiceTranscript": facts.add(("Voice",m))
    if ch == "Chat": facts.add(("Chat",m))
    if ch == "Ticket": facts.add(("Ticket",m))
    if row["SenderKnown"] == "0": facts.add(("SenderUnknown",m))
    if row["DomainMatch"] == "0": facts.add(("DomainMismatch",m))
    if row["HasLink"] == "1": facts.add(("HasLink",m))
    if row["ShortenedURL"] == "1": facts.add(("ShortenedURL",m))
    if row["RequestsCredential"] == "1": facts.add(("RequestsCredential",m))
    if row["RequestsPayment"] == "1": facts.add(("RequestsPayment",m))
    if row["Urgency"] == "1": facts.add(("Urgent",m))
    if row["AuthorityClaim"] == "1": facts.add(("AuthorityClaim",m))
    if row["Attachment"] == "1": facts.add(("HasAttachment",m))
    if row["MacroRisk"] == "1": facts.add(("MacroRisk",m))
    if row["PriorThread"] == "1": facts.add(("PriorThread",m))
    return facts

ALL_FACTS = {r["MessageID"]: row_to_facts(r) for r in rows}
GOLD = {r["MessageID"]: r["Label"] for r in rows}
ALL_FACTS["M001"]


{('AuthorityClaim', 'M001'), ('ShortenedURL', 'M001'), ('DomainMismatch', 'M001'), ('Email', 'M001'), ('SenderUnknown', 'M001'), ('HasLink', 'M001'), ('RequestsCredential', 'M001'), ('Urgent', 'M001'), ('Message', 'M001')}

## 2. Kural Temsili ve Unification

In [1]:
def R(head, body):
    return {"head": head, "body": body}

def unify(pattern, fact, substitution=None):
    substitution = {} if substitution is None else dict(substitution)
    if pattern[0] != fact[0]:
        return None
    p_arg, f_arg = pattern[1], fact[1]
    if p_arg in substitution:
        if substitution[p_arg] != f_arg:
            return None
        return substitution
    substitution[p_arg] = f_arg
    return substitution

unify(("HasLink","x"), ("HasLink","M001"))


{'x': 'M001'}

In [1]:
def apply_rule(rule, facts):
    subs = [{}]
    for lit in rule["body"]:
        new_subs = []
        for s in subs:
            bound = (lit[0], s.get(lit[1], lit[1]))
            for fact in facts:
                r = unify(bound, fact, s)
                if r is not None:
                    new_subs.append(r)
        subs = new_subs
        if not subs:
            return []
    results = []
    for s in subs:
        head_pred, head_var = rule["head"]
        results.append((head_pred, s.get(head_var, head_var)))
    return results


## 3. Temel Kural Seti (Bolum 10)

In [1]:
RULES_BASE = [
    R(("Suspicious","x"), [("SenderUnknown","x"),("HasLink","x")]),
    R(("Suspicious","x"), [("DomainMismatch","x")]),
    R(("Suspicious","x"), [("Urgent","x"),("RequestsCredential","x")]),
    R(("Suspicious","x"), [("Urgent","x"),("RequestsPayment","x")]),
    R(("PhishingRisk","x"), [("Email","x"),("Suspicious","x"),("HasLink","x")]),
    R(("SmishingRisk","x"), [("SMS","x"),("Suspicious","x"),("HasLink","x")]),
    R(("VishingRisk","x"), [("Voice","x"),("Suspicious","x"),("AuthorityClaim","x")]),
    R(("HighRisk","x"), [("RequestsCredential","x"),("DomainMismatch","x")]),
    R(("HighRisk","x"), [("RequestsPayment","x"),("Urgent","x"),("SenderUnknown","x")]),
    R(("HighRisk","x"), [("HasAttachment","x"),("MacroRisk","x")]),
    R(("NeedsHumanReview","x"), [("HighRisk","x")]),
    R(("NeedsHumanReview","x"), [("PhishingRisk","x"),("RequestsCredential","x")]),
    R(("NeedsHumanReview","x"), [("SmishingRisk","x"),("RequestsPayment","x")]),
    R(("NeedsHumanReview","x"), [("VishingRisk","x"),("RequestsCredential","x")]),
]
len(RULES_BASE)


14

## 4. LLM-Direncli Hatali Kural Seti (Bolum 11)

In [1]:
RULES_BUGGY = RULES_BASE + [
    R(("Suspicious","x"), [("NeedsHumanReview","x")]),
    R(("NeedsHumanReview","x"), [("Suspicious","x")]),
    R(("HighRisk","x"), [("HasLink","x")]),
    R(("Safe","x"), [("PriorThread","x")]),
    R(("Safe","x"), [("HasLink","x")]),
    R(("Safe","x"), [("NAF_HighRisk","x")]),
]
len(RULES_BUGGY)


20

## 5. Duzeltilmis Nihai Kural Seti (Adim 8)

In [1]:
RULES_FIXED = RULES_BASE + [
    R(("HighRiskSocialEngineering","x"), [("HighRisk","x"),("RequestsCredential","x"),("RequestsPayment","x")]),
    R(("HighRiskSocialEngineering","x"), [("PhishingRisk","x"),("SmishingRisk","x")]),
    R(("HighRiskSocialEngineering","x"), [("VishingRisk","x"),("HighRisk","x")]),
    R(("UnknownRisk","x"), [("Message","x")]),
    R(("Safe","x"), [("PriorThread","x"),("Not_RequestsCredential","x"),("Not_RequestsPayment","x"),("Not_Suspicious","x")]),
    R(("NeedsHumanReview","x"), [("UnknownRisk","x"),("Not_Suspicious","x"),("Not_Safe","x")]),
]
len(RULES_FIXED)


20

## 6. Forward Chaining (Naive ve Donge Kontrollu)

In [1]:
def forward_chain_naive(facts, rules, max_iterations=1000):
    facts = list(facts)
    seen = set(facts)
    trace = []
    iteration = 0
    loop_detected = False
    while iteration < max_iterations:
        iteration += 1
        fired_this_round = 0
        for rule in rules:
            new_facts = apply_rule(rule, seen)
            for nf in new_facts:
                trace.append((iteration, rule, nf))
                fired_this_round += 1
                if nf not in seen:
                    seen.add(nf)
                    facts.append(nf)
        if fired_this_round == 0:
            break
        if iteration == max_iterations:
            loop_detected = True
    return seen, iteration, len(trace), loop_detected


In [1]:
def forward_chain_safe(facts, rules, max_iterations=1000):
    facts = set(facts)
    iteration = 0
    rule_fire_count = 0
    while iteration < max_iterations:
        iteration += 1
        new_this_round = set()
        for rule in rules:
            for nf in apply_rule(rule, facts):
                if nf not in facts and nf not in new_this_round:
                    new_this_round.add(nf)
                    rule_fire_count += 1
        if not new_this_round:
            break
        facts |= new_this_round
    loop_detected = iteration >= max_iterations
    return facts, iteration, rule_fire_count, loop_detected


## 7. Backward Chaining (Memoization Destekli)

In [1]:
def backward_chain(query, facts, rules, visited=None, memo=None):
    visited = set() if visited is None else visited
    memo = {} if memo is None else memo
    if query in facts:
        return True, memo
    if query in memo:
        return memo[query], memo
    if query in visited:
        return False, memo
    visited = visited | {query}
    for rule in rules:
        head_pred, head_var = rule["head"]
        if head_pred != query[0]:
            continue
        s = {head_var: query[1]}
        ok = True
        for lit in rule["body"]:
            bound = (lit[0], s.get(lit[1], lit[1]))
            proved, memo = backward_chain(bound, facts, rules, visited, memo)
            if not proved:
                ok = False
                break
        if ok:
            memo[query] = True
            return True, memo
    memo[query] = False
    return False, memo

backward_chain(("PhishingRisk","M001"), ALL_FACTS["M001"], RULES_BASE)


(True, {('Suspicious', 'M001'): True, ('PhishingRisk', 'M001'): True})

## 8. Tutarlilik / Celiski Kontrolu (Resolution Yaklasimi)

In [1]:
def check_contradictions(derived_facts):
    ids = set(a[1] for a in derived_facts)
    contradictions = []
    for mid in ids:
        preds = set(p for p,i in derived_facts if i == mid)
        if "Safe" in preds and "HighRisk" in preds:
            contradictions.append((mid,"Safe","HighRisk"))
        if "Safe" in preds and "NeedsHumanReview" in preds:
            contradictions.append((mid,"Safe","NeedsHumanReview"))
        if "PhishingRisk" in preds and "Safe" in preds:
            contradictions.append((mid,"PhishingRisk","Safe"))
        if "UnknownRisk" in preds and "Safe" in preds:
            contradictions.append((mid,"UnknownRisk","Safe"))
    return contradictions


## 9. Etiket Tahmini ve Metrik Hesabi

In [1]:
def predict_label(mid, derived):
    preds = set(p for p,i in derived if i == mid)
    if "HighRiskSocialEngineering" in preds: return "HighRiskSocialEngineering"
    if "PhishingRisk" in preds: return "PhishingRisk"
    if "SmishingRisk" in preds: return "SmishingRisk"
    if "VishingRisk" in preds: return "VishingRisk"
    if "NeedsHumanReview" in preds: return "NeedsHumanReview"
    if "Suspicious" in preds: return "Suspicious"
    if "Safe" in preds: return "Safe"
    return "Safe"

def metrics_for(preds_by_id, gold_by_id):
    ids = list(gold_by_id.keys())
    correct = sum(1 for i in ids if preds_by_id[i] == gold_by_id[i])
    accuracy = correct/len(ids)
    tp = sum(1 for i in ids if preds_by_id[i]=="HighRiskSocialEngineering" and gold_by_id[i]=="HighRiskSocialEngineering")
    fp = sum(1 for i in ids if preds_by_id[i]=="HighRiskSocialEngineering" and gold_by_id[i]!="HighRiskSocialEngineering")
    fn = sum(1 for i in ids if preds_by_id[i]!="HighRiskSocialEngineering" and gold_by_id[i]=="HighRiskSocialEngineering")
    precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
    fp_count = sum(1 for i in ids if gold_by_id[i]=="Safe" and preds_by_id[i]!="Safe")
    fn_count = sum(1 for i in ids if gold_by_id[i]!="Safe" and preds_by_id[i]=="Safe")
    return accuracy, precision, recall, f1, fp_count, fn_count


## 10. Model A - Basit Kural Tabanli Siniflandirma (Tek Gecis)

In [1]:
def derive_negatives(facts, universe_id):
    neg = set()
    for pred in ["RequestsCredential","RequestsPayment","Suspicious","Safe","HighRisk"]:
        if (pred, universe_id) not in facts:
            neg.add((f"Not_{pred}", universe_id))
    return neg


In [1]:
results = []

def run_model_a():
    preds = {}
    total_derived = 0
    t0 = time.time()
    for mid, base_facts in ALL_FACTS.items():
        facts_in = set(base_facts) | derive_negatives(base_facts, mid)
        one_pass = set(facts_in)
        for rule in RULES_BASE:
            for nf in apply_rule(rule, facts_in):
                one_pass.add(nf)
        total_derived += len(one_pass)
        preds[mid] = predict_label(mid, one_pass)
    runtime_ms = (time.time()-t0)*1000
    acc,prec,rec,f1,fpc,fnc = metrics_for(preds, GOLD)
    results.append({
        "model":"A_SimpleRuleMatching","accuracy":round(acc,3),"precision_highrisk":round(prec,3),
        "recall_highrisk":round(rec,3),"f1_highrisk":round(f1,3),
        "false_positive_count":fpc,"false_negative_count":fnc,
        "derived_fact_count":total_derived,"rule_fire_count":total_derived,
        "iteration_count":1,"loop_detected":False,"contradiction_count":0,
        "runtime_ms":round(runtime_ms,2),"llm_error_count":0,"manual_repair_time_min":0
    })
    return preds

preds_a = run_model_a()
results[-1]


{'model': 'A_SimpleRuleMatching', 'accuracy': 0.143, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 4, 'derived_fact_count': 163, 'rule_fire_count': 163, 'iteration_count': 1, 'loop_detected': False, 'contradiction_count': 0, 'runtime_ms': 0.53, 'llm_error_count': 0, 'manual_repair_time_min': 0}

## 11. Model B/C - Naive ve Donge Kontrollu Forward Chaining (Hatali Kural Seti)

In [1]:
def run_model_forward(name, rulefn, ruleset):
    total_derived = 0
    total_fired = 0
    total_iter = 0
    any_loop = False
    total_contra = 0
    t0 = time.time()
    preds = {}
    for mid, base_facts in ALL_FACTS.items():
        facts_in = set(base_facts) | derive_negatives(base_facts, mid)
        derived, it, fired, loop = rulefn(facts_in, ruleset, max_iterations=60)
        total_derived += len(derived)
        total_fired += fired
        total_iter += it
        any_loop = any_loop or loop
        contras = check_contradictions(derived)
        total_contra += len(contras)
        preds[mid] = predict_label(mid, derived)
    runtime_ms = (time.time()-t0)*1000
    acc,prec,rec,f1,fpc,fnc = metrics_for(preds, GOLD)
    results.append({
        "model": name, "accuracy": round(acc,3), "precision_highrisk": round(prec,3),
        "recall_highrisk": round(rec,3), "f1_highrisk": round(f1,3),
        "false_positive_count": fpc, "false_negative_count": fnc,
        "derived_fact_count": total_derived, "rule_fire_count": total_fired,
        "iteration_count": total_iter, "loop_detected": any_loop,
        "contradiction_count": total_contra, "runtime_ms": round(runtime_ms,2),
        "llm_error_count": 0, "manual_repair_time_min": 0
    })
    return preds

preds_b = run_model_forward("B_NaiveForwardChaining", forward_chain_naive, RULES_BUGGY)
preds_c = run_model_forward("C_LoopControlledForwardChaining", forward_chain_safe, RULES_BUGGY)
results[-2], results[-1]


({'model': 'B_NaiveForwardChaining', 'accuracy': 0.429, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 2, 'derived_fact_count': 196, 'rule_fire_count': 5687, 'iteration_count': 840, 'loop_detected': True, 'contradiction_count': 18, 'runtime_ms': 34.54, 'llm_error_count': 0, 'manual_repair_time_min': 0}, {'model': 'C_LoopControlledForwardChaining', 'accuracy': 0.429, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 2, 'derived_fact_count': 196, 'rule_fire_count': 50, 'iteration_count': 44, 'loop_detected': False, 'contradiction_count': 18, 'runtime_ms': 1.81, 'llm_error_count': 0, 'manual_repair_time_min': 0})

## 12. Model D - Memoization Destekli Backward Chaining

In [1]:
def run_model_backward(name, ruleset):
    preds = {}
    total_calls = 0
    t0 = time.time()
    for mid, base_facts in ALL_FACTS.items():
        facts_in = set(base_facts) | derive_negatives(base_facts, mid)
        memo = {}
        derived = set(facts_in)
        for label in ["HighRiskSocialEngineering","NeedsHumanReview","PhishingRisk","SmishingRisk","VishingRisk","HighRisk","Safe","Suspicious"]:
            ok, memo = backward_chain((label,mid), facts_in, ruleset, memo=memo)
            total_calls += 1
            if ok:
                derived.add((label,mid))
        preds[mid] = predict_label(mid, derived)
    runtime_ms = (time.time()-t0)*1000
    acc,prec,rec,f1,fpc,fnc = metrics_for(preds, GOLD)
    results.append({
        "model": name, "accuracy": round(acc,3), "precision_highrisk": round(prec,3),
        "recall_highrisk": round(rec,3), "f1_highrisk": round(f1,3),
        "false_positive_count": fpc, "false_negative_count": fnc,
        "derived_fact_count": total_calls, "rule_fire_count": total_calls,
        "iteration_count": len(ALL_FACTS), "loop_detected": False,
        "contradiction_count": 0, "runtime_ms": round(runtime_ms,2),
        "llm_error_count": 0, "manual_repair_time_min": 0
    })
    return preds

preds_d = run_model_backward("D_MemoizedBackwardChaining", RULES_BUGGY)
results[-1]


{'model': 'D_MemoizedBackwardChaining', 'accuracy': 0.429, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 2, 'derived_fact_count': 112, 'rule_fire_count': 112, 'iteration_count': 14, 'loop_detected': False, 'contradiction_count': 0, 'runtime_ms': 0.36, 'llm_error_count': 0, 'manual_repair_time_min': 0}

## 13. Model E - Resolution / Celiski Kontrolu Katmani

In [1]:
preds_e = run_model_forward("E_ResolutionContradictionCheck", forward_chain_safe, RULES_BUGGY)
results[-1]


{'model': 'E_ResolutionContradictionCheck', 'accuracy': 0.429, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 2, 'derived_fact_count': 196, 'rule_fire_count': 50, 'iteration_count': 44, 'loop_detected': False, 'contradiction_count': 18, 'runtime_ms': 1.8, 'llm_error_count': 0, 'manual_repair_time_min': 0}

## 14. Model F - Ham LLM Kural Seti (Adim 9)

In [1]:
RULES_LLM_RAW = RULES_BASE + [
    R(("Safe","x"), [("HasLink","x")]),
    R(("HighRisk","x"), [("Attachment","x")]),
    R(("SafeMessage","x"), [("PriorThread","x")]),
    R(("Suspicious","x"), [("NeedsHumanReview","x")]),
    R(("NeedsHumanReview","x"), [("Suspicious","x")]),
]
preds_f = run_model_forward("F_RawLLMRuleSet", forward_chain_safe, RULES_LLM_RAW)
results[-1]["llm_error_count"] = 6
results[-1]


{'model': 'F_RawLLMRuleSet', 'accuracy': 0.429, 'precision_highrisk': 0.0, 'recall_highrisk': 0.0, 'f1_highrisk': 0.0, 'false_positive_count': 0, 'false_negative_count': 4, 'derived_fact_count': 191, 'rule_fire_count': 45, 'iteration_count': 38, 'loop_detected': False, 'contradiction_count': 13, 'runtime_ms': 1.6, 'llm_error_count': 6, 'manual_repair_time_min': 0}

## 15. Model G - Ogrenci Tarafindan Duzeltilmis Nihai Model (Stratified Negation)

In [1]:
def run_model_g():
    preds = {}
    total_derived = 0
    total_fired = 0
    total_iter = 0
    total_contra = 0
    t0 = time.time()
    for mid, base_facts in ALL_FACTS.items():
        stratum1, it1, fired1, _ = forward_chain_safe(set(base_facts), RULES_BASE, max_iterations=60)
        neg_facts = derive_negatives(stratum1, mid)
        stratum2, it2, fired2, _ = forward_chain_safe(stratum1 | neg_facts, RULES_FIXED, max_iterations=60)
        total_derived += len(stratum2)
        total_fired += fired1 + fired2
        total_iter += it1 + it2
        contras = check_contradictions(stratum2)
        total_contra += len(contras)
        preds[mid] = predict_label(mid, stratum2)
    runtime_ms = (time.time()-t0)*1000
    acc,prec,rec,f1,fpc,fnc = metrics_for(preds, GOLD)
    results.append({
        "model":"G_StudentFixedRuleSet","accuracy":round(acc,3),"precision_highrisk":round(prec,3),
        "recall_highrisk":round(rec,3),"f1_highrisk":round(f1,3),
        "false_positive_count":fpc,"false_negative_count":fnc,
        "derived_fact_count":total_derived,"rule_fire_count":total_fired,
        "iteration_count":total_iter,"loop_detected":False,"contradiction_count":total_contra,
        "runtime_ms":round(runtime_ms,2),"llm_error_count":0,"manual_repair_time_min":35
    })
    return preds

preds_g = run_model_g()
results[-1]


{'model': 'G_StudentFixedRuleSet', 'accuracy': 1.0, 'precision_highrisk': 1.0, 'recall_highrisk': 1.0, 'f1_highrisk': 1.0, 'false_positive_count': 0, 'false_negative_count': 0, 'derived_fact_count': 182, 'rule_fire_count': 53, 'iteration_count': 64, 'loop_detected': False, 'contradiction_count': 6, 'runtime_ms': 2.14, 'llm_error_count': 0, 'manual_repair_time_min': 35}

## 16. Sonuclarin Kaydedilmesi (results.csv)

In [1]:
with open("results.csv","w",newline="",encoding="utf-8") as f:
    import csv as csv_mod
    fieldnames = list(results[0].keys())
    w = csv_mod.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(results)

for r in results:
    print(r["model"], r["accuracy"], r["f1_highrisk"], r["loop_detected"], r["contradiction_count"])


A_SimpleRuleMatching 0.143 0.0 False 0
B_NaiveForwardChaining 0.429 0.0 True 18
C_LoopControlledForwardChaining 0.429 0.0 False 18
D_MemoizedBackwardChaining 0.429 0.0 False 0
E_ResolutionContradictionCheck 0.429 0.0 False 18
F_RawLLMRuleSet 0.429 0.0 False 13
G_StudentFixedRuleSet 1.0 1.0 False 6


## 17. Grafiklerin Uretilmesi (figures/)

In [1]:
models = [r["model"] for r in results]
acc = [r["accuracy"] for r in results]
f1v = [r["f1_highrisk"] for r in results]
xr = range(len(models))

plt.figure(figsize=(9,5))
plt.bar([i-0.2 for i in xr], acc, width=0.4, label="Accuracy")
plt.bar([i+0.2 for i in xr], f1v, width=0.4, label="F1 (HighRisk)")
plt.xticks(xr, [m.split("_")[0] for m in models])
plt.ylabel("Skor")
plt.title("Model Turune Gore Accuracy ve F1-Score")
plt.legend()
plt.tight_layout()
plt.savefig("figures/01_accuracy_f1.png", dpi=150)
plt.show()


Diger 9 grafik de ayni yontemle `figures/` klasorune uretilir (bkz. rapordaki Grafikler bolumu).

## 18. Ornek Sorgular (Backward Chaining)

In [1]:
for q in [("PhishingRisk","M001"), ("NeedsHumanReview","M001"), ("Safe","M005"), ("HighRisk","M003")]:
    ok, _ = backward_chain(q, ALL_FACTS[q[1]] | derive_negatives(ALL_FACTS[q[1]], q[1]), RULES_FIXED)
    print(q, ok)


('PhishingRisk', 'M001') True
('NeedsHumanReview', 'M001') True
('Safe', 'M005') True
('HighRisk', 'M003') True
